# Parse quote PDFs

Extract text from quote/order PDFs using `pdfplumber` (same parser as the price list notebook).

## Setup

Install `pdfplumber` in the kernel if needed: run `%pip install pdfplumber` once.

In [40]:
from pathlib import Path
import pdfplumber

# Path to the quote PDF
PDF_PATH = Path("2026/VP Quotes 2026/PO#_7FITGERALD_Order_518771.PDF")

## Open PDF and list pages

In [41]:
with pdfplumber.open(PDF_PATH) as pdf:
    print(f"Pages: {len(pdf.pages)}")
    for i, page in enumerate(pdf.pages):
        print(f"  {i + 1}: {page.width} x {page.height}")

Pages: 3
  1: 612 x 792
  2: 612 x 792
  3: 612 x 792


## Extract text (all pages)

Combine into a single string with page dividers (`--- Page N ---`).

In [42]:
parts = []
with pdfplumber.open(PDF_PATH) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text()
        parts.append(f"--- Page {i + 1} ---\n\n{text or '(no text)'}")
full_text = "\n\n".join(parts)
print(f"Combined text: {len(full_text)} chars, {len(parts)} pages")

Combined text: 4489 chars, 3 pages


## Save to file (optional)

In [43]:
# out_path = PDF_PATH.with_suffix(".txt")
# with open(out_path, "w", encoding="utf-8") as f:
#     f.write(full_text)
# print(f"Wrote {out_path}")

## Parse to extracted_data JSON shape
Run the parser on `full_text` to get a list of line items (same structure as `extracted_data.json`).

In [44]:
import importlib
import util.parse_quote_text
importlib.reload(util.parse_quote_text)
from util.parse_quote_text import parse_quote_text
import json

parsed = parse_quote_text(full_text, PDF_PATH.name)
print(f"Parsed {len(parsed)} line items")
print("\n--- JSON output ---")
print(json.dumps(parsed, indent=2))

TypeError: reload() argument must be a module

In [ ]:
with open("parsed_json.json", "w", encoding="utf-8") as f:
    f.write(json.dumps(parsed, indent=2))

In [ ]:
import pandas as pd
# Flatten for a quick table: one row per line item, key fields
rows = []
for item in parsed:
    for u in item.get("units", []):
        rows.append({
            "line_no": item["line_no"],
            "combo_type": item["combo_type"],
            "description": u.get("description", ""),
            "width_in": u.get("width_in"),
            "height_in": u.get("height_in"),
            "sq_ft": u.get("sq_ft"),
            "base_price": u.get("base_price"),
            "colour_in": item.get("colour_in", ""),
            "colour_out": item.get("colour_out", ""),
            "line_total": item.get("line_total"),
        })
df = pd.DataFrame(rows)
display(df)

,line_no,combo_type,description,width_in,height_in,sq_ft,base_price,colour_in,colour_out,line_total
0,1,V-F/CS-R,VINYL FIXED,68.1250,56.625,28.19,415.28,WHT,WHT,1080.56
1,1,V-F/CS-R,CASEMENT RIGHT,24.0000,56.625,9.67,209.86,WHT,WHT,1080.56
2,2,CS-L/V-F/CS-R,CASMENT LEFT,20.0000,63.125,8.89,188.12,WHT,WHT,1212.93
3,2,CS-L/V-F/CS-R,VINYL FIXED,52.1250,63.125,24.00,353.56,WHT,WHT,1212.93
4,2,CS-L/V-F/CS-R,CASEMENT RIGHT,20.0000,63.125,8.89,188.12,WHT,WHT,1212.93
5,3,CS-L/V-F,CASMENT LEFT,24.8125,63.375,11.56,209.86,WHT,WHT,672.75
6,3,CS-L/V-F,VINYL FIXED,24.8125,63.375,11.56,170.30,WHT,WHT,672.75
7,4,CS-L/V-F,CASMENT LEFT,24.9375,50.625,9.39,209.86,WHT,WHT,603.41
8,4,CS-L/V-F,VINYL FIXED,24.9375,50.625,9.39,138.33,WHT,WHT,603.41
9,5,CS-L/V-F,CASMENT LEFT,28.8125,50.625,10.83,209.86,WHT,WHT,649.36


## Parse all PDFs in VP Quotes 2026 (single combined JSON)

Iterate over every PDF in `2026/VP Quotes 2026`, extract text, parse, and append to one list; then save to a single JSON file.

In [ ]:
VP_QUOTES_DIR = Path("2026/VP Quotes 2026")
OUT_ALL_JSON = Path("2026/parsed_all_vp_quotes_2026.json")

pdf_paths = sorted(VP_QUOTES_DIR.glob("*.PDF")) or sorted(VP_QUOTES_DIR.glob("*.pdf"))
combined = []
for pdf_path in pdf_paths:
    with pdfplumber.open(pdf_path) as pdf:
        parts = [f"--- Page {i + 1} ---\n\n{page.extract_text() or '(no text)'}" for i, page in enumerate(pdf.pages)]
    full_text = "\n\n".join(parts)
    parsed = parse_quote_text(full_text, pdf_path.name)
    combined.extend(parsed)
    print(f"  {pdf_path.name}: {len(parsed)} line(s)")
OUT_ALL_JSON.write_text(json.dumps(combined, indent=2), encoding="utf-8")
print(f"\nWrote {len(combined)} line items to {OUT_ALL_JSON}")

  PO#_14FIELDING_Order_519307.PDF: 3 line(s)
  PO#_14FIELDING_Order_519310.PDF: 18 line(s)
  PO#_161SPROULE_Order_519243.PDF: 1 line(s)
  PO#_201LITTLEWOO_Order_519086 (1).PDF: 22 line(s)
  PO#_201LITTLEWOO_Order_519086.PDF: 22 line(s)
  PO#_201RIDGE_Order_518776.PDF: 12 line(s)
  PO#_227AVDELL_Order_519107.PDF: 1 line(s)
  PO#_271KING_Order_519306.PDF: 0 line(s)
  PO#_276BLACKTH_Order_519367.PDF: 1 line(s)
  PO#_2BERTRAM_Order_519398.PDF: 3 line(s)
  PO#_3275MEAD_Order_519109.PDF: 1 line(s)
  PO#_39PARK_Order_519237.PDF: 1 line(s)
  PO#_49DANIELLE_Order_518694.PDF: 3 line(s)
  PO#_504BUSH_Order_518690.PDF: 3 line(s)
  PO#_521WELLINGTO_Order_519265.PDF: 1 line(s)
  PO#_5338HILTON_Order_519366.PDF: 1 line(s)
  PO#_70GLENBURN_Order_519311.PDF: 1 line(s)
  PO#_7FITGERALD_Order_518771.PDF: 6 line(s)
  PO#_STK101225_Order_518692.PDF: 16 line(s)

Wrote 116 line items to 2026/parsed_all_vp_quotes_2026.json


In [ ]:
# Optional: summary table of combined data (one row per line item)
rows = []
for item in combined:
    for u in item.get("units", []):
        rows.append({
            "source_file": item["source_file"],
            "line_no": item["line_no"],
            "combo_type": item["combo_type"],
            "description": u.get("description", ""),
            "width_in": u.get("width_in"),
            "height_in": u.get("height_in"),
            "line_total": item.get("line_total"),
        })
pd.DataFrame(rows)

,source_file,line_no,combo_type,description,width_in,height_in,line_total
0,PO#_14FIELDING_Order_519307.PDF,1,CS-L/V-F,CASMENT LEFT,22.6875,69.0000,665.17
1,PO#_14FIELDING_Order_519307.PDF,1,CS-L/V-F,VINYL FIXED,22.6875,69.0000,665.17
2,PO#_14FIELDING_Order_519307.PDF,2,,HALF ROUND,57.3750,28.6875,1287.53
3,PO#_14FIELDING_Order_519307.PDF,2,,VINYL FIXED,57.3750,70.9375,1287.53
4,PO#_14FIELDING_Order_519307.PDF,3,V-F/CS-R,VINYL FIXED,22.6875,69.0000,665.17
...,...,...,...,...,...,...,...
144,PO#_STK101225_Order_518692.PDF,12,CS-L,CASMENT LEFT,30.0000,30.0000,235.42
145,PO#_STK101225_Order_518692.PDF,13,CS-L,CASMENT LEFT,30.0000,30.0000,235.42
146,PO#_STK101225_Order_518692.PDF,14,CS-R,CASEMENT RIGHT,30.0000,30.0000,235.42
147,PO#_STK101225_Order_518692.PDF,15,CS-R,CASEMENT RIGHT,30.0000,30.0000,235.42
